# 预设拆分 / 回收工具

对应文件: `预设/本格修仙Kemini5-3.4.json`

- **expand()**: 将 `prompts` 数组中每一项的 `content` 写入 `预设/拆分/{name}.txt`，同时生成 `overview.txt` 把所有 content 顺序拼接。
- **collapse()**: 读取 `预设/拆分/{name}.txt` 中的内容，写回 JSON 对应条目的 `content`，覆盖原值后保存。

重名条目以 `__1` `__2` 后缀区分；含 Windows 非法字符的文件名会被替换为 `_`。

In [1]:
from pathlib import Path
import json
import re

ROOT = Path('..').resolve()
JSON_PATH = ROOT / '预设' / '本格修仙Kemini5-3.4.json'
OUT_DIR = ROOT / '预设' / '拆分'
OVERVIEW = OUT_DIR / 'overview.txt'

ILLEGAL = re.compile(r'[\\/:*?"<>|\r\n\t]')
SEP = '\n\n' + '=' * 60 + '\n'

print('JSON :', JSON_PATH)
print('OUT  :', OUT_DIR)

JSON : D:\application\Tavern\settings\git\世界书\Cultivation-Card-Game\预设\本格修仙Kemini5-3.4.json
OUT  : D:\application\Tavern\settings\git\世界书\Cultivation-Card-Game\预设\拆分


In [2]:
def _safe_name(name: str) -> str:
    s = ILLEGAL.sub('_', name).strip().rstrip('. ')
    return s or '_unnamed'


def _build_filenames(prompts):
    """按 prompts 顺序生成与每个条目一一对应的文件名（不含扩展名）。"""
    used = {}
    names = []
    for p in prompts:
        base = _safe_name(p.get('name', ''))
        n = used.get(base, 0)
        used[base] = n + 1
        names.append(base if n == 0 else f'{base}__{n}')
    return names


def _load_json():
    return json.loads(JSON_PATH.read_text(encoding='utf-8'))

In [3]:
def expand():
    data = _load_json()
    prompts = data.get('prompts', [])
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    filenames = _build_filenames(prompts)

    written = 0
    overview_parts = []
    for fname, p in zip(filenames, prompts):
        content = p.get('content', '') or ''
        (OUT_DIR / f'{fname}.txt').write_text(content, encoding='utf-8', newline='\n')
        overview_parts.append(f'# {p.get("name", "")}\n{content}')
        written += 1

    OVERVIEW.write_text(SEP.join(overview_parts), encoding='utf-8', newline='\n')
    print(f'[expand] 写入 {written} 个 txt -> {OUT_DIR}')
    print(f'[expand] overview.txt 大小: {OVERVIEW.stat().st_size} bytes')

In [4]:
def collapse():
    data = _load_json()
    prompts = data.get('prompts', [])
    filenames = _build_filenames(prompts)

    updated = 0
    missing = []
    for fname, p in zip(filenames, prompts):
        fp = OUT_DIR / f'{fname}.txt'
        if not fp.exists():
            missing.append(fname)
            continue
        new_content = fp.read_text(encoding='utf-8')
        if p.get('content', '') != new_content:
            p['content'] = new_content
            updated += 1

    JSON_PATH.write_text(
        json.dumps(data, ensure_ascii=False, indent=4),
        encoding='utf-8',
        newline='\n',
    )
    print(f'[collapse] 更新 {updated} / {len(prompts)} 条 prompt')
    if missing:
        print(f'[collapse] 缺失 {len(missing)} 个 txt（已跳过）: {missing[:5]}{"..." if len(missing) > 5 else ""}')

## 展开

In [5]:
expand()

[expand] 写入 100 个 txt -> D:\application\Tavern\settings\git\世界书\Cultivation-Card-Game\预设\拆分
[expand] overview.txt 大小: 140317 bytes


## 收回 (写回 JSON)

In [ ]:
collapse()